# Chargement des données disponibles en ligne

## Setup

In [ ]:
import numpy as np
import pandas as pd
import requests
import duckdb
import os
import s3fs

# Create filesystem object
S3_ENDPOINT_URL = "https://" + os.environ["AWS_S3_ENDPOINT"]
fs = s3fs.S3FileSystem(client_kwargs={'endpoint_url': S3_ENDPOINT_URL})
BUCKET = "mathildedonnenfeld/diffusion/bistrots_et_bulletins"

## Tables municipales

### Lecture

In [ ]:
url_t1 = "https://www.data.gouv.fr/api/1/datasets/r/4feeef01-24f7-4d5a-914f-8aa806f31ec2"
t1_muni_26 = pd.read_csv(url_t1, sep=";")
t1_muni_26.head()

In [ ]:
url_t2 = "https://www.data.gouv.fr/api/1/datasets/r/6ff67a28-01bf-459e-beca-dd7aa8132dc1"
t2_muni_26 = pd.read_csv(url_t2, sep=";")
t2_muni_26.head()

### Écriture dans minio

In [ ]:
FILE_KEY_OUT_S3 = "t1_muni.csv"
FILE_PATH_OUT_S3 = BUCKET + "/" + FILE_KEY_OUT_S3

with fs.open(FILE_PATH_OUT_S3, 'w') as file_out:
    t1_muni_26.to_csv(file_out)

In [ ]:
FILE_KEY_OUT_S3 = "t2_muni.csv"
FILE_PATH_OUT_S3 = BUCKET + "/" + FILE_KEY_OUT_S3

with fs.open(FILE_PATH_OUT_S3, 'w') as file_out:
    t2_muni_26.to_csv(file_out)

## Base Sirene

*Note : Les difficultés de connexion récurrentes à la base Sirene historique - qui s'ajoutent à celles causées par la limitation de l'API de data.gouv.fr, nous ont poussées à télécharger celle-ci manuellement et à la stocker dans Minio. C'est ce chargement auquel nous nous réferrons pour l'ensemble du projet, mais il est identique à la base disponible en ligne à l'url commentée dans la cellule ci-dessous en date du 21/10/26.* 

In [ ]:
#url_sirene_historique = "https://www.data.gouv.fr/api/1/datasets/r/2b3a0c79-f97b-46b8-ac02-8be6c1f01a8c"
sirene_historique_local = "s3://mathildedonnenfeld/diffusion/bistrots_et_bulletins/stock-stocketablissementhistorique-parquet.parquet"

In [ ]:
# On récupère l'état de chaque bistrot dans sa période active au 1er mars 2026
etat_bistrots_sirene_muni = duckdb.sql(f"""
    SELECT siret
    FROM read_parquet('{sirene_historique_local}')
    WHERE activitePrincipaleEtablissement == '56.30Z'
    AND etatAdministratifEtablissement == 'A'
    AND dateDebut < '2026-03-01'
    AND (dateFin >= '2026-03-01' OR dateFin IS NULL)
""").df()

In [ ]:
etat_bistrots_sirene_muni.shape[0]

In [ ]:
etat_bistrots_sirene_muni.siret.nunique()

In [ ]:
df_sirets = etat_bistrots_sirene_muni.siret

In [ ]:
etat_bistrots_sirene_muni

In [ ]:
# On récupère dans le répertoire stock actuel les établissements qui étaient un débit de boisson ouvert au 1er mars 2026

url_sirene = "https://www.data.gouv.fr/api/1/datasets/r/a29c1297-1f92-4e2a-8f6b-8c902ce96c5f"

bistrots_sirene = duckdb.sql(f"""
    SELECT *
    FROM read_parquet('{url_sirene}')
    JOIN etat_bistrots_sirene_muni USING (siret)
""").df()
bistrots_sirene.head()


# Note : La variable du code commune s'appelle ici codeCommuneEtablissement.

In [ ]:
bistrots_sirene.shape[0]

In [ ]:
bistrots_sirene.siret.nunique()

In [ ]:
bistrots_sirene.etatAdministratifEtablissement.describe()

In [ ]:
49295-47227

In [ ]:
bistrots_sirene_fermetures = duckdb.sql(f"""
    SELECT siret, dateDebut AS date_fermeture
    FROM read_parquet('{sirene_historique_local}')
    WHERE etatAdministratifEtablissement = 'F'
    AND changementEtatAdministratifEtablissement = 'true'
    AND activitePrincipaleEtablissement == '56.30Z'
    AND dateDebut >= '2026-03-01'
""").df()

In [ ]:
bistrots_sirene_fermetures.siret.nunique()
# Il manque (2144-2068)=76 sirets qui ont fermé dans l'intervalle alors qu'ils n'on pas été comptés en mars. On considère la méthode de sélection de période plus fiable que celle du décompte des fermetures, c'est donc cette première que l'on conserve. 

In [ ]:
# Alternative en attendant de récupérer l'historique 
bistrots_sirene_alt = tous_bistrots_sirene[(tous_bistrots_sirene.etatAdministratifEtablissement == "A") & (tous_bistrots_sirene.dateCreationEtablissement < "2026-03-01")]
bistrots_sirene_alt.head()
print(bistrots_sirene_alt.shape[0])
print(bistrots_sirene.shape[0])
# Ok, dans la version alternative il nous manque tous les établissements qui ont fermé depuis les municipales, que l'on veut prendre en compte dans l'analyse.

In [ ]:
FILE_KEY_OUT_S3 = "bistrots_sirene.csv"
FILE_PATH_OUT_S3 = BUCKET + "/" + FILE_KEY_OUT_S3

with fs.open(FILE_PATH_OUT_S3, 'w') as file_out:
    bistrots_sirene.to_csv(file_out)

In [ ]:
#Récupération des données Filosofi au niveau commune

def get_melodi(dataset, dimension, code, value_name, time_period, geo_level="COM"):

  url = f"https://api.insee.fr/melodi/data/{dataset}"

  params = {
    "GEO": geo_level,
    dimension: code,
    "TIME_PERIOD": time_period,
    "maxResult": 40000,
  }

  response = requests.get(url, params=params, timeout=60)
  response.raise_for_status()
  observations = response.json()["observations"]

  return pd.DataFrame(
    {
      "CODGEO": obs["dimensions"]["GEO"].split("-")[-1],
      value_name: obs["measures"]["OBS_VALUE_NIVEAU"].get("value"),
    }
    for obs in observations
  )

niveau_vie = get_melodi("DS_FILOSOFI_CC", "FILOSOFI_MEASURE", "MED_SL", "NIVVIE_MEDIAN", 2023)
taux_pauvrete = get_melodi("DS_FILOSOFI_CC", "FILOSOFI_MEASURE", "PR_MD60", "TAUX_PAUVRETE", 2023)
population = get_melodi("DS_POPULATIONS_HISTORIQUES", "POPREF_MEASURE", "PMUN", "POPULATION", 2023)

url_cog_2023 = "https://www.insee.fr/fr/statistiques/fichier/6800675/v_commune_2023.csv"
url_backup = "https://minio.lab.sspcloud.fr/lgaliana/data/python-ENSAE/cog_2023.csv"

try:
  cog_2023 = pd.read_csv(url_cog_2023)
except requests.exceptions.Timeout:
  cog_2023 = pd.read_csv(url_backup)

noms_communes = (
  cog_2023
  .loc[cog_2023["TYPECOM"] == "COM", ["COM", "LIBELLE"]]
  .rename(columns={"COM": "CODGEO", "LIBELLE": "LIBGEO"})
)

filosofi = (
  niveau_vie
  .merge(taux_pauvrete, on="CODGEO", how="outer")
  .merge(population, on="CODGEO", how="outer")
  .merge(noms_communes, on="CODGEO", how="left")
  [["CODGEO", "LIBGEO", "POPULATION", "NIVVIE_MEDIAN", "TAUX_PAUVRETE"]]
)

filosofi["POPULATION"] = filosofi["POPULATION"].astype(int)
filosofi["dep"] = filosofi["CODGEO"].str[:2]

In [ ]:
filosofi.head()

In [ ]:
FILE_KEY_OUT_S3 = "filosofi.csv"
FILE_PATH_OUT_S3 = BUCKET + "/" + FILE_KEY_OUT_S3

with fs.open(FILE_PATH_OUT_S3, 'w') as file_out:
    filosofi.to_csv(file_out)

In [ ]:
fs.ls("mathildedonnenfeld/diffusion/bistrots_et_bulletins/")